# Session 2: the boiling-point curve from physics
**A. Real-world problem.** Session 1 read boiling points from a table. Where do the table
numbers come from, and what valve weight holds a cooker at 100 kPa above the air?

**Safety and scope.** Educational model only. Never modify, block or remove the valves of
real equipment.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R = 8.314             # gas constant, J/(mol K)
L_MOL = 40700.0       # molar latent heat of vaporisation of water near 100 C, J/mol (about 40.7 kJ/mol)
P_ATM = 101.325       # kPa
T_REF = 373.15        # K, boiling point at P_ATM
G = 9.81              # m/s^2

# Approximate steam-table values (kPa absolute, degrees C) to compare with.
TABLE = [(50.0, 81.3), (101.325, 100.0), (200.0, 120.2), (300.0, 133.5), (400.0, 143.6)]

## Step 1: Clausius-Clapeyron
For a liquid in equilibrium with its vapour, treating the vapour as an ideal gas and the
latent heat L as constant:

    ln(P2 / P1) = (L / R) * (1/T1 - 1/T2)        (T in kelvin, any pressure unit)

Take (P1, T1) = (101.325 kPa, 373.15 K). Solving for T2 gives the boiling point at P2.

In [ ]:
def boiling_point_k(p_abs_kpa, latent=L_MOL):
    """Boiling point in kelvin at an absolute pressure (kPa), from Clausius-Clapeyron."""
    if p_abs_kpa <= 0:
        raise ValueError("absolute pressure must be positive")
    return 1.0 / (1.0 / T_REF - R * np.log(p_abs_kpa / P_ATM) / latent)


def boiling_point_c(p_abs_kpa, latent=L_MOL):
    return float(boiling_point_k(p_abs_kpa, latent) - 273.15)


def vapour_pressure_kpa(temp_c, latent=L_MOL):
    """The inverse: pressure (kPa absolute) at which water boils at temp_c."""
    return float(P_ATM * np.exp(latent / R * (1.0 / T_REF - 1.0 / (temp_c + 273.15))))

## Step 2: gauge versus absolute, and the Q10 rule of thumb
A gauge reads pressure above the atmosphere: P_abs = P_gauge + 101.325 kPa. The physics
(boiling point) needs the ABSOLUTE pressure.

Rule of thumb for cooking: rate roughly doubles per 10 degrees C, so
rate factor = 2^((T - 100) / 10). It is only a rule of thumb.

In [ ]:
def gauge_to_abs(p_gauge_kpa):
    return p_gauge_kpa + P_ATM


def rate_factor(temp_c, q10=2.0):
    return float(q10 ** ((temp_c - 100.0) / 10.0))

## Step 3: the weight on the valve (design)
A weighted valve sits on a small opening of diameter d. It lifts, and steam escapes, when the
steam pressure force on the opening equals the weight:

    P_gauge * A = m * g       with  A = pi d^2 / 4

so P_gauge = m g / A, and for a wanted gauge pressure the mass is m = P_gauge * A / g.

In [ ]:
def valve_gauge_kpa(mass_kg, diameter_m):
    area = np.pi * diameter_m ** 2 / 4.0
    return float(mass_kg * G / area / 1000.0)


def valve_mass_kg(p_gauge_kpa, diameter_m):
    area = np.pi * diameter_m ** 2 / 4.0
    return float(p_gauge_kpa * 1000.0 * area / G)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    pa = np.linspace(40.0, 420.0, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(pa, [boiling_point_c(x) for x in pa], label="Clausius-Clapeyron (L = 40.7 kJ/mol)")
    ax.plot([p for p, _ in TABLE], [t for _, t in TABLE], "o", label="approximate steam table")
    ax.set_xlabel("absolute pressure (kPa)")
    ax.set_ylabel("boiling point (degrees C)")
    ax.set_title("Boiling point against pressure")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "clausius_clapeyron.png", dpi=120)

    gauge = np.linspace(0.0, 250.0, 100)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    for d in (3.0, 4.0, 5.0):
        ax2.plot([valve_mass_kg(g, d / 1000.0) * 1000.0 for g in gauge], gauge, label=f"opening {d:.0f} mm")
    ax2.axhline(100.0, color="red", ls="--", label="100 kPa gauge target")
    ax2.set_xlabel("valve mass (g)")
    ax2.set_ylabel("gauge pressure held (kPa)")
    ax2.set_title("Valve design: mass needed for a gauge pressure")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "valve_design.png", dpi=120)

    print("P_abs (kPa)   CC (C)   table (C)   error (C)")
    for p, t in TABLE:
        print(f"{p:9.1f}   {boiling_point_c(p):7.2f}   {t:7.1f}   {boiling_point_c(p) - t:+7.2f}")
    t100 = boiling_point_c(gauge_to_abs(100.0))
    print(f"gauge 100 kPa -> absolute {gauge_to_abs(100.0):.1f} kPa -> {t100:.2f} C, rate x{rate_factor(t100):.2f}")
    print(f"mass for 100 kPa gauge on a 4 mm opening: {valve_mass_kg(100.0, 0.004) * 1000:.1f} g")

## Where this stops
L is not exactly constant and steam is not exactly ideal, so errors grow far from 100 C.
Session 3 uses this boiling-point law inside a full heating and cooking simulation.

## Coding tasks
1. **Easy.** Use `boiling_point_c` to print the boiling point at 70 kPa (about 3000 m height) and at 300 kPa; predict both before you run it.
2. **Stretch.** Try the latent heat `latent=38000` and `latent=44000` in `boiling_point_c(200)` and compare with 120.2; say which is closer and why L is only approximately constant.
3. **Explain.** Write `valve_mass_for_temperature(temp_c, diameter_m)` that finds the valve mass for a wanted boiling temperature, using `vapour_pressure_kpa`; check it for 120 degrees C.

## Assessment
- **A. Conceptual.** Why must the Clausius-Clapeyron relation use absolute temperature in kelvin and absolute pressure, not the gauge reading?
- **B. Numerical.** Use Clausius-Clapeyron to find the boiling point at 50 kPa absolute (for example high on a mountain). Compare with the approximate table value 81.3 degrees C.
- **C. Graph and simulation interpretation.** From the first figure, in which range of pressure does the Clausius-Clapeyron curve and the steam-table points agree best, and where do they differ most? Suggest why.
- **D. Design and optimisation challenge.** Design: a cooker has a 5 mm valve opening. Choose a valve mass that holds 100 kPa gauge, say how the gauge pressure changes if the mass is 10 percent too heavy, and give the new boiling point.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("at 101.325 kPa the boiling point is 100 C", abs(L.boiling_point_c(101.325) - 100.0) < 1e-9))
    t100g = L.boiling_point_c(L.gauge_to_abs(100.0))
    checks.append((f"worked example: 100 kPa gauge (201.325 kPa absolute) boils at {t100g:.1f} C, about 120.6",
                   abs(t100g - 120.6) < 0.05 and abs(L.rate_factor(t100g) - 4.17) < 0.02))
    checks.append(("near 200 kPa the result is within 0.5 C of the steam table (120.2 C)", abs(L.boiling_point_c(200.0) - 120.2) < 0.5))
    errs = [abs(L.boiling_point_c(p) - t) for p, t in L.TABLE]
    checks.append(("all table points are within 0.6 C", max(errs) < 0.6))
    checks.append(("vapour_pressure_kpa undoes boiling_point_c", abs(L.vapour_pressure_kpa(L.boiling_point_c(250.0)) - 250.0) < 1e-6))
    # hand calculation: 1/T2 = 1/373.15 - ln(2) R / L when the pressure doubles
    inv = 1 / 373.15 - np.log(2.0) * 8.314 / 40700.0
    checks.append(("hand calculation: doubling the pressure gives 1/T2 = 1/373.15 - ln2*R/L", abs(L.boiling_point_k(2 * 101.325) - 1 / inv) < 1e-6))
    checks.append(("boiling point rises with pressure", L.boiling_point_c(150.0) < L.boiling_point_c(250.0)))
    checks.append(("rate factor doubles per 10 C", abs(L.rate_factor(110.0) - 2.0) < 1e-12))
    m = L.valve_mass_kg(100.0, 0.004)
    checks.append((f"design: 100 kPa gauge on a 4 mm opening needs {m * 1000:.1f} g (about 128)", abs(m * 1000 - 128.1) < 0.2))
    checks.append(("valve mass and pressure formulas are inverses", abs(L.valve_gauge_kpa(m, 0.004) - 100.0) < 1e-9))
    try:
        L.boiling_point_k(0.0)
        edge = False
    except ValueError:
        edge = True
    checks.append(("zero absolute pressure raises ValueError", edge))
    checks.append(("a larger opening needs a heavier valve for the same pressure", L.valve_mass_kg(100.0, 0.005) > L.valve_mass_kg(100.0, 0.004)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
